[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/surfaces/smoke_rings/smoke_rings.ipynb)

# Smoke Rings

This example moves thin vortex filaments, such as the core of a smoke ring, through an ideal fluid. In Euler flow, the flow of a fluid without viscosity that cannot be compressed, vorticity gathered in a thin tube along a curve moves the fluid everywhere around it, by the Biot–Savart law, and the fluid carries the tube along. The largest part of that motion comes from close to each point of the curve: it moves the point across the curve, in the direction perpendicular both to the curve and to the way it bends, at a speed set by how sharply it bends. Keeping that part alone is the smoke ring flow, the local induction approximation: a circle flies along its axis without changing shape, any other curve writhes as it goes, and Hoffmann gives the flow for polygons with edges of unit length and calls the surface the curve sweeps over time its Hashimoto surface. Keeping the whole Biot–Savart law lets separate rings move each other: two smoke rings one behind the other take turns passing through each other, leapfrogging.

The computation is written in the geometric algebra of three-dimensional space. The velocity of each vertex comes straight from the turn between its two edges: the product of the outgoing edge with the incoming one.

In [ ]:
# The repository root on the path, for numga and the examples; in Colab, fetch the repository first.
import sys
from pathlib import Path

if "google.colab" in sys.modules:
    root = Path("/content/numga")
    if not root.exists():
        import subprocess
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/EelcoHoogendoorn/numga.git", str(root)], check=True)
else:
    root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "numga").is_dir() and (p / "examples").is_dir())
sys.path.insert(0, str(root))

In [ ]:
%matplotlib inline
import numpy as np

from numga import NumpyContext, concatenate, stack
from numga.algebras import VGA3D as ga
from examples.animation import save_figure, show_animation
from examples.surfaces.smoke_rings import render
from examples.surfaces.smoke_rings.core import closed, ring, torus_knot, wavy_ring

In [ ]:
mv = NumpyContext(ga).multivector
Scalar = ga.gatype.scalar()
Vector = ga.gatype.vector()
Bivector = ga.gatype.bivector()
Rotor = ga.gatype.rotor()
Pseudoscalar = ga.gatype.pseudoscalar()
# The unit volume that no motion changes, intrinsic: a product with it is a complement.
unit_volume = mv.antiscalar([1.0])
# The filament's circulation changes sign under a mirror: an intrinsic scalar, set to one here.
Circulation = Scalar.derive.flipped_orientation
circulation = mv(Circulation, [1.0])

## 1. The turn at a vertex

At a vertex of a polygon, between the incoming edge and the outgoing one, the product of the outgoing edge with the incoming edge is the turn from one to the other: its scalar part, the product of the lengths plus `outgoing | incoming`, and its plane, `outgoing ^ incoming`. The plane over the scalar part is the tangent of half the turning angle times the unit plane of the turn, and its dual is the direction perpendicular to that plane, the binormal of the polygon at the vertex. Twice that, over the mean length of the two edges, is the velocity of the vertex: on edges of unit length it is exactly Hoffmann's, and for short edges it tends to the curvature times the binormal.

A regular polygon turns by the same angle at every vertex, about the same axis, so every vertex moves alike, along the axis: the ring flies without changing shape, at twice the tangent of half its turn per vertex.

In the notation of the paper the velocity reads $\dot\gamma_k = 2\, S_k \times S_{k-1} / (1 + \langle S_k, S_{k-1}\rangle)$ for edges $S$ of unit length, and in the smooth limit $\dot\gamma = \kappa B$, the curvature times the Frenet binormal.

In [ ]:
def edges(points: Vector) -> Vector:
    """The edges of closed polygons `[..., n]`, edge k from point k to point k + 1."""
    return concatenate([points[..., 1:], points[..., :1]], axis=-1) - points    # [..., n] Vector


def velocity(points: Vector, circulation: Circulation) -> Vector:
    """Twice the dual of the turn's Cayley parameter, times the circulation, over the mean edge length."""
    outgoing = edges(points)                                                     # [..., n] Vector
    incoming = concatenate([outgoing[..., -1:], outgoing[..., :-1]], axis=-1)    # [..., n] Vector
    # The turn from the incoming edge to the outgoing one: its scalar part, and its plane.
    lengths = outgoing.norm() * incoming.norm()                                  # [..., n] Scalar
    halving = 2 * (outgoing ^ incoming).dual() * circulation / (lengths + (outgoing | incoming))   # [..., n] Vector
    return halving / ((outgoing.norm() + incoming.norm()) / 2)                   # [..., n] Vector


# A regular polygon of unit edges.
ring_count = 60
flying = velocity(ring(ring_count), circulation)                                 # [n] Vector
print("spread of the ring's velocities:", (flying - flying[0]).norm().to_array().max())
print("speed along the axis, and twice the tangent of the half turn:", (flying | mv.z).to_array()[0], 2 * np.tan(np.pi / ring_count))

## 2. What the flow keeps, and the surface it sweeps

Each vertex moves perpendicular to both of its edges, so no edge changes its length, whatever the lengths are. The flow keeps the polygon's vector area too, the sum of each point wedged with the next, which is the impulse of a vortex filament. The flow is integrated by classical Runge–Kutta steps. A ring carrying a standing wave of three crests, its edges tilted up and down by the sine of three times their angle around the axis, flies along its axis while the waves travel around it; joining the ring at successive times by strips gives the surface it sweeps, its Hashimoto surface, coloured from early to late.

In [ ]:
def impulse(points: Vector) -> Bivector:
    """The vector area of closed polygons, twice over: the sum of each point wedged with the next."""
    return (points ^ (points + edges(points))).sum(axis=-1)                     # [...] Bivector


def flow(points: Vector, rate, step: float, steps: int, every: int):
    """A flow of polygons with the velocity `rate`, by classical Runge–Kutta steps: the polygons every `every`
    steps, and at the end."""
    for count in range(steps + 1):
        if count % every == 0 or count == steps:
            yield points
        first = rate(points)
        second = rate(points + first * (step / 2))
        third = rate(points + second * (step / 2))
        fourth = rate(points + third * step)
        points = points + (first + 2 * second + 2 * third + fourth) * (step / 6)


# The wavy ring's vertices, the waves around it and the tilt of its edges; the flow's step, its number of
# steps, and the steps between kept rings.
wavy = wavy_ring(90, 3, 0.4)                                                     # [n] Vector
surface = stack(list(flow(wavy, lambda points: velocity(points, circulation), 0.4, 750, 10)))                                  # [times, n] Vector
lengths = edges(surface).norm().to_array()                                       # [times, n]
areas = impulse(surface).kernel                                                  # [times, 3]
print("largest change of an edge's length:", np.abs(lengths - lengths[0]).max())
print("largest change of the impulse, relative:", np.abs(areas - areas[0]).max() / np.abs(areas[0]).max())
save_figure(render.draw_surface(closed(surface)), "smoke_rings_surface");

## 3. A knotted vortex

A vortex filament tied in a trefoil knot, on a torus about the axis, turns about the axis as it flies along it, and keeps its knotted shape while its loops ripple: the flow moves each part of the knot by its own bending, and the trefoil's three loops bend alike. The knot's edges are shorter than one and not all equal, so its step is shorter, and its velocity, divided by the mean edge length, is the curvature times the binormal rather than Hoffmann's unit-edge form. The curve is shaded from far to near.

In [ ]:
# The trefoil's windings around the axis and around the tube, the tube's radii and its vertices; the flow's
# step, its number of steps, the steps between frames, and the duration of a frame.
knot = torus_knot(2, 3, 6.0, 2.5, 240)                                           # [n] Vector
curves = stack(list(flow(knot, lambda points: velocity(points, circulation), 0.05, 2400, 40)))                                 # [times, n] Vector
knot_lengths = edges(curves).norm().to_array()                                   # [times, n]
print("largest change of an edge's length:", np.abs(knot_lengths - knot_lengths[0]).max())
show_animation(render.animate_curves(closed(curves)), "smoke_rings_knot", 60)

## 4. Discrete time

Time can be made discrete as well, by Hoffmann's Bäcklund transformation. A new polygon is the old one with an offset of one fixed length at every vertex, and with every new edge again of unit length. Each offset follows from the one before by a fold: across an edge, the offset is turned about the diagonal of the quad between the old edge and the new one, `edge - offset`. With the transformation's parameter as the volume `twist`, a multiple of the pseudoscalar, a vector plus that volume, times the pseudoscalar, is a scalar plus a bivector: a rotor about the vector, so the fold is a single sandwich, `((edge - offset - twist) * unit_volume).normalized() >> offset`, with `unit_volume` the unit pseudoscalar that no motion changes. The parameter sets how far the fold turns, by twice the angle whose tangent is the diagonal's length over the parameter. The turn keeps each new edge's part along the diagonal, so every new edge is exactly as long as the old.

On the sphere of unit offsets, folding across a unit edge keeps the edge's two directions fixed, and turns and scales the sphere around them alike for every edge. That is a Lorentz transformation of the sphere, taken as the sphere of light rays of spacetime: a boost along the edge and a turn about it. In the algebra of space a Lorentz transformation acts on paravectors, a scalar plus a vector, by the sandwich with the reverse, and the null paravector `1 + offset` stands for an offset. Across an edge it is `(-(1 + 2 * twist.inverse()).log() * edge / 2).exp()`: the real part of the logarithm is the boost's rapidity and its volume part the turn's angle. Folding once around a closed polygon is the product of these rotors. The boosts in a long product grow from edge to edge until round-off swamps the offsets, so the offsets themselves are folded one after another, and the product serves where a whole lap is meant.

For a closed polygon the offsets must come round to where they started: carried once around the lap, a start at minus the last edge lands on the lap's attracting fixed point. With offsets of unit length, the new polygon lies one vertex back along the old one, moved by the smoke ring flow over a time of about half the parameter, so relabelled by one vertex a transformation is a step in time. Every edge keeps exactly unit length, step after step, and the ring, a discrete elastic curve, moves exactly rigidly. The surface the wavy ring sweeps this way, in steps of a quarter unit of time, is the discrete Hashimoto surface, every one of its quads folded about its diagonal.

In the quaternions of the paper, with the vectors as imaginary quaternions, the fold is the Riccati-type equation $(\lambda + \tilde S)(\lambda + \operatorname{Re}\nu + v) = (\lambda + \operatorname{Re}\nu + v_{+})(\lambda + S)$ for all $\lambda$, with $\tilde S$ the new edge and $v$, $v_{+}$ the offsets at its ends, and the Lorentz transformation the matrix of $SL(2, \mathbb{C})$ that acts on the sphere by the Möbius map.

In [ ]:
def fold(edge: Vector, offset: Vector, twist: Pseudoscalar) -> Vector:
    """The next offset: the current one turned about the quad's diagonal, by the rotor a vector plus a volume is times the pseudoscalar."""
    return ((edge - offset - twist) * unit_volume).normalized() >> offset        # [...] Vector


def backlund(points: Vector, start: Vector, twist: Pseudoscalar) -> Vector:
    """One Bäcklund transformation of polygons `[..., n]`, from the offset at the first vertex."""
    sides = points[..., 1:] - points[..., :-1]                                   # [..., n - 1] Vector

    def offsets():
        # Every offset is the one before, folded across the edge between them.
        offset = start
        yield offset
        for index in range(sides.shape[-1]):
            offset = fold(sides[..., index], offset, twist)
            yield offset

    return points + stack(list(offsets()), axis=-1)                              # [..., n] Vector


def periodic(points: Vector, twist: Pseudoscalar) -> Vector:
    """The unit offset at the first vertex of closed polygons of unit edges that comes round to itself."""
    sides = edges(points)                                                        # [..., n] Vector
    # Every fold as a Lorentz transformation about its edge, and the lap as their product.
    folds = (-(1 + 2 * twist.inverse()).log() * sides / 2).exp()                 # [..., n] Lorentz rotor
    lap = folds.cumprod(axis=-1)[..., -1]                                        # [...] Lorentz rotor
    # Minus the last edge, carried once around the lap, lands on its attracting fixed point; the
    # paravector's bivector and volume parts are zero.
    image = lap * (1 - sides[..., -1]) * lap.reverse()                           # [...] null paravector
    return image.cast(Vector) / image.cast(Scalar)                               # [...] Vector


def advance(points: Vector, twist: Pseudoscalar) -> Vector:
    """One step in discrete time: the transformation with periodic offsets, relabelled by one vertex."""
    return backlund(closed(points), periodic(points, twist), twist)[..., 1:]     # [..., n] Vector


# The transformation's parameter, as an oriented volume.
twist = mv.xyz * 0.5                                                             # [] Pseudoscalar

# The fold, and the Lorentz transformation of the same edge, on a few unit offsets.
edge = mv.vector([0, 0, 1])                                                      # [] Vector
offsets = mv.vector([[1, 0, 0], [0, 1, 0], [0, -0.6, 0.8]])                      # [3] Vector
rotor = (-(1 + 2 * twist.inverse()).log() * edge / 2).exp()
image = rotor * (1 + offsets) * rotor.reverse()
lorentz = image.cast(Vector) / image.cast(Scalar)                                 # [3] Vector
print("largest difference between the fold and the Lorentz transformation:", np.abs((lorentz - fold(edge, offsets, twist)).kernel).max())

# The rings' vertices and the steps; the plain ring and the wavy one stepped together.
count, steps = 40, 100
rings = stack([ring(count), wavy_ring(count, 3, 0.4)])                           # [rings, n] Vector


def stepped():
    curves = rings
    yield curves
    for _ in range(steps):
        curves = advance(curves, twist)
        yield curves


discrete = stack(list(stepped()))                                                # [steps + 1, rings, n] Vector
discrete_lengths = edges(discrete).norm().to_array()                             # [steps + 1, rings, n]
moved_ring = discrete[-1, 0]                                                     # [n] Vector
print("largest departure of an edge from unit length:", np.abs(discrete_lengths - 1).max())
print("spread of the ring's heights after the steps:", np.ptp((moved_ring | mv.z).to_array()))
save_figure(render.draw_surface(closed(discrete[:, 1])), "smoke_rings_discrete");

## 5. A loop running along a straight vortex

A soliton is a solitary wave that travels without changing its shape. On a straight vortex line it is a loop that runs along the line and turns about it as it goes, Hasimoto's soliton, and one Bäcklund transformation of a straight line makes one. Every fold across a straight line keeps the line's two directions fixed: offsets near the forward direction move away from it, and offsets near the backward direction move towards it. An offset leaning just off the forward direction stays near it for a while, then crosses over to the backward one, and the new polygon carries a loop where the crossing happens. Every new vertex lies within one offset of the line, so the offset's length sets the loop's size, and the parameter sets how drawn out it is along the line.

The steps of section 4 carry the loop along. On an open line the offsets start at minus the first edge, which the straight end keeps exactly, so relabelled by one vertex the end stays where it is; the line runs on straight past its last vertex. The turn at each vertex, the outgoing edge times the incoming one, carries the loop's shape: its angle is the turning of the tangent there, and through the loop the tangent turns once around, the angles adding up to a full turn at every step whatever the loop's size. The animation draws the line as it is at each step and, behind it, the surface it has swept so far: the straight parts do not move and sweep nothing, and the loop, turning about the line as it runs along it, sweeps a screw-shaped tube. Drawn along the line and through the steps, the turning angles are a streak of one width moving at one speed.

In the notation of the theory of curves, Hasimoto's map $\psi = \kappa\, e^{i\int\tau\, ds}$ of the curvature $\kappa$ and the torsion $\tau$ takes the smoke ring flow to the nonlinear Schrödinger equation $i\psi_t + \psi_{ss} + \tfrac{1}{2}|\psi|^2\psi = 0$, and the soliton's curvature is $\kappa = 2\nu\operatorname{sech}(\nu s)$, whose integral is $2\pi$ for every $\nu$.

In [ ]:
def turns(points: Vector) -> Rotor:
    """The turn at every inner vertex of polygons of unit edges: the outgoing edge times the incoming one."""
    sides = points[..., 1:] - points[..., :-1]                                   # [..., n - 1] Vector
    return sides[..., 1:] * sides[..., :-1]                                      # [..., n - 2] Rotor


# A straight line of unit edges; how far the first offset leans off the forward direction, its length and
# the parameter; the parameter of the steps that carry the loop, and their number.
line_count, lean, reach = 75, 1e-5, 4.0
loop_twist, carry_twist = mv.xyz * 2.0, mv.xyz * 1.0                             # [] Pseudoscalar
carry_steps = 90
line = mv.x * (np.arange(line_count) - line_count / 2)                           # [n] Vector
start = ((mv.zx * lean).exp() >> mv.x) * reach                                   # [] Vector


def carried():
    curve = backlund(line, start, loop_twist)                                   # [n] Vector
    yield curve
    for _ in range(carry_steps):
        # Offsets from minus the first edge, relabelled by one vertex; the line runs on straight past its end.
        moved = backlund(curve, curve[0] - curve[1], carry_twist)[1:]           # [n - 1] Vector
        curve = concatenate([moved, 2 * moved[-1:] - moved[-2:-1]])             # [n] Vector
        yield curve


solitons = stack(list(carried()))                                               # [steps + 1, n] Vector
angles = turns(solitons).log().norm().to_array()                                # [steps + 1, n - 2]
print("total turning through the loop at the first and the last step, in full turns:", angles[[0, -1]].sum(axis=-1) / (2 * np.pi))

In [ ]:
# The steps between frames.
show_animation(render.animate_trail(solitons[::2]), "smoke_rings_soliton", 60)
save_figure(render.draw_turns(turns(solitons)), "smoke_rings_turns");

## 6. Two rings leapfrogging

The whole Biot–Savart law moves every vertex by every segment of every filament, not by its own two edges alone. A straight segment of circulation `circulation` moves a point by the dual of the plane spanned by the segment and the point's separation from the segment's middle, over the cube of the separation's length, softened by a core radius so that the law stays finite on the filament itself, the vortex filament of Rosenhead and Moore. Two equal rings, one a little behind the other on the same axis, move each other: the front ring widens and slows, the rear one narrows and speeds up, passes through the front one, and the two change places, twice in every cycle. A ring alone flies along its axis without changing shape, and the rings' impulse, each ring's vector area times its circulation, is kept. The tubes each ring sweeps through two cycles are drawn in their own colours, the half nearer the viewer cut away: wherever one ring passes through the other, its tube runs inside.

In the notation of fluid dynamics the law reads $u(x) = \frac{\Gamma}{4\pi} \oint \frac{d\ell \times (x - y)}{(|x - y|^2 + \delta^2)^{3/2}}$, for a filament of circulation $\Gamma$ and core radius $\delta$.

In [ ]:
def induced(points: Vector, circulations: Circulation, core_radius: float) -> Vector:
    """The velocity at every vertex of polygonal vortex filaments `[rings, n]` induced by every segment of
    every filament: the dual of the plane of the segment and the separation, over its softened cube."""
    segments = edges(points)                                                     # [rings, n] Vector
    separations = points.reshape(-1)[:, None] - (points + segments / 2).reshape(-1)[None, :]   # [rings n, rings n] Vector
    softened = separations.norm_squared() + core_radius**2                       # [rings n, rings n] Scalar
    strengths = circulations[..., None].broadcast_to(points.shape).reshape(-1) / (4 * np.pi)   # [rings n] Circulation
    pulls = (segments.reshape(-1) ^ separations).dual() * strengths / (softened * softened.square_root())
    return pulls.sum(axis=1).reshape(points.shape)                               # [rings, n] Vector


# Two equal rings on one axis, one behind the other: their radius, vertices and heights, their circulations
# and core radius; the flow's step and number of steps, through two cycles.
radius, count = 5.0, 48
rings = (ring(count) * (radius * 2 * np.sin(np.pi / count)))[None, :] + mv.z * np.array([0.0, 2.5])[:, None]   # [rings, n] Vector
circulations, core_radius = mv(Circulation, [[10.0], [10.0]]), 0.4
frogs = stack(list(flow(rings, lambda points: induced(points, circulations, core_radius), 0.4, 50, 1)))   # [times, rings, n] Vector
heights = (frogs | mv.z).to_array().mean(axis=-1)                                # [times, rings]
radii = (frogs - mv.z * (frogs | mv.z)).norm().to_array().mean(axis=-1)          # [times, rings]
passes = np.flatnonzero(np.diff(np.sign(heights[:, 1] - heights[:, 0])))
print("times at which one ring passes through the other:", passes * 0.4)
save_figure(render.draw_swept(closed(frogs)), "smoke_rings_swept");

In [ ]:
show_animation(render.animate_curves(closed(frogs)), "smoke_rings_leapfrog", 60)

In [ ]:
# checks
# The ring flies rigidly along its axis at twice the tangent of half its turn per vertex.
np.testing.assert_allclose((flying - flying[0]).norm().to_array(), 0.0, atol=1e-13)
np.testing.assert_allclose(np.abs((flying | mv.z).to_array()), 2 * np.tan(np.pi / ring_count), rtol=1e-12)
# Both flows keep every edge's length; the wavy ring keeps its impulse.
np.testing.assert_allclose(lengths, np.broadcast_to(lengths[0], lengths.shape), atol=1e-9)
np.testing.assert_allclose(knot_lengths, np.broadcast_to(knot_lengths[0], knot_lengths.shape), atol=1e-9)
np.testing.assert_allclose(areas, np.broadcast_to(areas[0], areas.shape), atol=1e-6 * np.abs(areas[0]).max())
# In discrete time every edge keeps unit length, and the ring moves rigidly.
np.testing.assert_allclose(discrete_lengths, 1.0, atol=1e-12)
np.testing.assert_allclose((moved_ring | mv.z).to_array(), (moved_ring[0] | mv.z).to_array(), atol=1e-12)
# The soliton keeps unit edges, and the tangent turns once around through the loop at every step; the
# tolerance allows for the loop's tails beyond the line's ends.
np.testing.assert_allclose((solitons[:, 1:] - solitons[:, :-1]).norm().to_array(), 1.0, atol=1e-12)
np.testing.assert_allclose(angles.sum(axis=-1), 2 * np.pi, atol=1e-3)
# The leapfrogging rings take turns, the narrower one passing through the wider.
behind = np.argmin(heights[passes], axis=-1)
assert len(passes) >= 4 and np.all(radii[passes, behind] < radii[passes, 1 - behind])